# re_nilm — Full Library Demo

This notebook covers every major capability of the `re_nilm` library in four sections:

1. **Single customer** — load data, run all detectors, inspect results
2. **Train models** — retrain AC / HP classifiers from Dataport labels
3. **Small portfolio** — run the pipeline on N customers, plot detection rates
4. **Full portfolio** — load or run all customers, plot aggregate results

Each section is self-contained. Sections 2–4 adapt to what data is available.

In [ ]:
import sys
from pathlib import Path

# Allow running from the notebooks/ directory without reinstalling the package
REPO = Path().resolve().parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR     = REPO / "data" / "re_data" / "ETHZ_ALL"   # 15-min customer parquets
WEATHER_CACHE = REPO / "data" / "out" / "weather_cache.parquet"
RESULTS_DIR  = REPO / "data" / "processed" / "out"      # pipeline outputs
RESULTS_FILE = RESULTS_DIR / "results_all_customers.parquet"

# Existing reference outputs from prior runs (used in Section 4 if full run not done)
REF_CAPACITY  = REPO / "data" / "out" / "capacity_autosave.parquet"
REF_INDICATORS = REPO / "data" / "out" / "pv_indicators_clean.parquet"

print("Data dir exists:", DATA_DIR.exists())
print("Weather cache:", WEATHER_CACHE.exists())
print("Full results:", RESULTS_FILE.exists())

In [ ]:
# ── Load or fetch weather (cached after first run) ─────────────────────────────
if WEATHER_CACHE.exists():
    weather = pd.read_parquet(WEATHER_CACHE)
    print(f"Weather loaded from cache: {len(weather):,} rows, "
          f"{weather.dt_utc.min().date()} → {weather.dt_utc.max().date()}")
else:
    from re_nilm.data.loaders.weather import load_meteoswiss
    WEATHER_CACHE.parent.mkdir(parents=True, exist_ok=True)
    print("Fetching MeteoSwiss weather data (takes ~30s)...")
    weather = load_meteoswiss(cache_path=WEATHER_CACHE)
    print(f"Weather fetched: {len(weather):,} rows")

---
## 1  Single Customer

Load one customer's time series and run every detector individually.
This is the lowest-level API — no pipeline, no config file.

In [ ]:
from re_nilm.pipeline.customer_index import build_customer_file_index, load_customer_from_index

index = build_customer_file_index(DATA_DIR)
all_ids = list(index.keys())
print(f"{len(all_ids):,} customers indexed")

# Pick a customer known to have PV (from reference data if available)
if REF_CAPACITY.exists():
    ref_cap = pd.read_parquet(REF_CAPACITY)
    pv_ids = ref_cap.loc[ref_cap.pv_capacity_kwp > 5, "customer_id"].tolist()
    DEMO_ID = next((c for c in pv_ids if c in index), all_ids[1])
else:
    DEMO_ID = all_ids[1]

customer_df = load_customer_from_index(DEMO_ID, index)
print(f"Customer: {DEMO_ID[:12]}...  rows: {len(customer_df):,}")
print(f"Date range: {customer_df.DT_UTC.min().date()} → {customer_df.DT_UTC.max().date()}")
customer_df.head(3)

In [ ]:
# ── Run all detectors ──────────────────────────────────────────────────────────
from re_nilm.detectors.pv import PVDetector
from re_nilm.estimators.pv_capacity import PVCapacityEstimator
from re_nilm.detectors.battery import BatteryDetector
from re_nilm.estimators.battery_capacity import BatteryCapacityEstimator
from re_nilm.detectors.ev import EVDetector
from re_nilm.estimators.ev_sessions import EVSessionEstimator

pv_det = PVDetector(corr_threshold=0.3, min_yearly_prod_kwh=1.0)
pv_cap = PVCapacityEstimator(bootstrap_n=200)
batt_det = BatteryDetector(enforce_pv_required=True)
batt_cap = BatteryCapacityEstimator()
ev_det = EVDetector(prob_threshold=0.3)
ev_sess = EVSessionEstimator()

pv_result   = pv_det.predict_customer(customer_df, weather)
cap_result  = pv_cap.estimate(customer_df, pv_result, weather) if pv_result else None
batt_result = batt_det.predict_customer(customer_df, weather, pv_result=cap_result)
batt_cap_r  = batt_cap.estimate(customer_df, {**(batt_result or {}), "pv_result": cap_result}, weather)
ev_result   = ev_det.predict_customer(customer_df, weather)
ev_sessions = ev_sess.estimate(customer_df, ev_result, weather) if ev_result else None

print("PV  :", pv_result)
print("Cap :", {k: round(v, 2) for k, v in (cap_result or {}).items() if isinstance(v, float)})
print("Batt:", batt_result)
print("EV  :", ev_result)

In [ ]:
# ── AC / HP detectors (skip gracefully if model files missing) ─────────────────
from re_nilm.detectors.ac import ACDetector
from re_nilm.detectors.heat_pump import HeatPumpDetector

ac_path = REPO / "models" / "ac_detector_v1.joblib"
hp_path = REPO / "models" / "hp_detector_v1.joblib"

ac_result = hp_result = None
if ac_path.exists():
    ac_det = ACDetector.load(ac_path)
    ac_result = ac_det.predict_customer(customer_df, weather)
    print("AC :", ac_result)
else:
    print("AC detector model not found — train it first (Section 2)")

if hp_path.exists():
    hp_det = HeatPumpDetector.load(hp_path)
    hp_result = hp_det.predict_customer(customer_df, weather)
    print("HP :", hp_result)
else:
    print("HP detector model not found — train it first (Section 2)")

In [ ]:
# ── Plot: load profile for one week in July ────────────────────────────────────
df = customer_df.copy()
df["DT_UTC"] = pd.to_datetime(df["DT_UTC"])
df = df.set_index("DT_UTC").sort_index()

week = df.loc["2024-07-01":"2024-07-07"]
w_week = weather.set_index("dt_utc").loc["2024-07-01":"2024-07-07"]

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

ax = axes[0]
ax.fill_between(week.index, week["CONSO_KWH"] * 4, alpha=0.4, label="Consumption (kW)", color="steelblue")
if "PROD_KWH" in week.columns:
    ax.fill_between(week.index, week["PROD_KWH"] * 4, alpha=0.6, label="Export (kW)", color="orange")
ax.set_ylabel("Power (kW)")
ax.legend(loc="upper right")

# Annotate detected appliances
labels = []
if pv_result and pv_result.get("has_pv"):
    kwp = (cap_result or {}).get("pv_capacity_kwp", 0)
    labels.append(f"PV ✓  {kwp:.1f} kWp")
if batt_result and batt_result.get("has_battery"):
    labels.append("Battery ✓")
if ev_result and ev_result.get("has_ev"):
    labels.append(f"EV ✓  {ev_result.get('ev_total_sessions', 0)} sessions")
if ac_result and ac_result.get("has_ac"):
    labels.append("AC ✓")
if hp_result and hp_result.get("has_hp"):
    labels.append(f"HP ✓  {hp_result.get('hp_type', '')}")
if labels:
    ax.set_title(f"Customer {DEMO_ID[:8]}...   |   " + "   ".join(labels))
else:
    ax.set_title(f"Customer {DEMO_ID[:8]}...   (no appliances detected)")

ax2 = axes[1]
ax2.fill_between(w_week.index, w_week["global_rad_W"], alpha=0.5, color="gold", label="Irradiance (W/m²)")
ax2.set_ylabel("W/m²")
ax2.legend(loc="upper right")

fig.tight_layout()
plt.show()

In [ ]:
# ── EV charging sessions (if detected) ────────────────────────────────────────
if ev_sessions and "ev_sessions" in ev_sessions:
    sess_df = ev_sessions["ev_sessions"]
    print(f"{len(sess_df)} charging sessions detected")
    display(sess_df.head(10))

    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].hist(sess_df["duration_h"].dropna(), bins=15, color="steelblue", edgecolor="white")
    axes[0].set_xlabel("Session duration (h)")
    axes[0].set_ylabel("Count")
    axes[1].hist(sess_df["energy_kWh"].dropna(), bins=15, color="orange", edgecolor="white")
    axes[1].set_xlabel("Session energy (kWh)")
    fig.suptitle("EV charging session distribution", fontsize=12)
    fig.tight_layout()
    plt.show()
elif ev_result:
    print(f"EV prob={ev_result['prob_ev']:.2f} — below threshold or no detailed sessions")

---
## 2  Train Models

AC and HP detectors are Random Forest classifiers trained on Dataport households with sub-metering.
This section shows how to run training. **Skip if you don't have `all_sources_load_with_weather.parquet`**.

In [ ]:
TRAINING_DATA = REPO / "data" / "processed_data" / "all_sources_load_with_weather.parquet"
MODELS_DIR    = REPO / "models"
MODELS_DIR.mkdir(exist_ok=True)

if not TRAINING_DATA.exists():
    print("Training data not found at:", TRAINING_DATA)
    print("Expected: all_sources_load_with_weather.parquet — raw 15-min time series")
    print("Columns: [type, source, dt_utc, glob_rad, value_kw_mean, id_customer, temp]")
    print("Skipping training — models will be loaded from models/ if present.")
else:
    df_train = pd.read_parquet(TRAINING_DATA)
    is_raw = "type" in df_train.columns and "value_kw_mean" in df_train.columns
    print(f"Training data: {len(df_train):,} rows, {df_train['id_customer'].nunique():,} customers")
    print("Format:", "raw time series" if is_raw else "pre-computed feature table")
    print("Columns:", df_train.columns.tolist())
    if is_raw:
        print("Type breakdown:", df_train["type"].value_counts().to_dict())

In [ ]:
# Train AC detector
# The trainer auto-detects raw time series and runs feature extraction internally.
if TRAINING_DATA.exists():
    from re_nilm.training.trainers.ac_detector_trainer import ACDetectorTrainer

    ac_trainer = ACDetectorTrainer(test_size=0.2)
    ac_trainer.fit(df_train)
    metrics = ac_trainer.evaluate()
    print(f"AC detector — F1: {metrics['f1']:.3f}  (test n={metrics['n_test']})")  
    print(metrics.get("report", ""))

    ac_trainer.save(MODELS_DIR / "ac_detector_v1.joblib")
    print("Saved:", MODELS_DIR / "ac_detector_v1.joblib")
else:
    print("Skipping AC training (training data not found)")

In [ ]:
# Train HP detector
# The trainer auto-detects raw time series and runs feature extraction internally.
if TRAINING_DATA.exists():
    from re_nilm.training.trainers.hp_detector_trainer import HPDetectorTrainer

    hp_trainer = HPDetectorTrainer(test_size=0.2)
    hp_trainer.fit(df_train)
    metrics = hp_trainer.evaluate()
    print(f"HP detector — F1: {metrics['f1']:.3f}  (test n={metrics['n_test']})")  
    print(metrics.get("report", ""))

    hp_trainer.save(MODELS_DIR / "hp_detector_v1.joblib")
    print("Saved:", MODELS_DIR / "hp_detector_v1.joblib")
else:
    print("Skipping HP training (training data not found)")

---
## 3  Small Portfolio (N customers)

Run all heuristic detectors over a subset of customers using the library classes directly.
This gives immediate feedback without a full pipeline run.

In [ ]:
from tqdm.auto import tqdm

N_CUSTOMERS = 200  # change to taste
sample_ids = all_ids[:N_CUSTOMERS]

pv_det   = PVDetector(corr_threshold=0.3, min_yearly_prod_kwh=1.0)
pv_cap   = PVCapacityEstimator(bootstrap_n=100)   # fewer bootstraps for speed
batt_det = BatteryDetector(enforce_pv_required=True)
ev_det   = EVDetector(prob_threshold=0.3)

rows = []
for cid in tqdm(sample_ids, desc="Detecting"):
    df = load_customer_from_index(cid, index)
    if df.empty:
        continue

    pv  = pv_det.predict_customer(df, weather) or {"customer_id": cid, "has_pv": False, "prob_pv": 0.0}
    cap = pv_cap.estimate(df, pv, weather) if pv.get("has_pv") else {}
    bat = batt_det.predict_customer(df, weather, pv_result=cap) or {}
    ev  = ev_det.predict_customer(df, weather) or {}

    row = {"customer_id": cid}
    row.update({k: v for k, v in pv.items() if k != "customer_id"})
    row.update({k: v for k, v in cap.items() if k != "customer_id"})
    row.update({k: v for k, v in bat.items() if k not in row})
    row.update({k: v for k, v in ev.items() if k not in row})
    rows.append(row)

sample_results = pd.DataFrame(rows)
print(f"Processed {len(sample_results)} customers")
sample_results[["customer_id", "has_pv", "pv_capacity_kwp", "has_battery", "has_ev", "prob_ev"]].head(8)

In [ ]:
# ── Detection rates ────────────────────────────────────────────────────────────
n_total = len(sample_results)
rates = {
    "PV":      sample_results["has_pv"].sum()      if "has_pv"      in sample_results.columns else 0,
    "Battery": sample_results["has_battery"].sum()  if "has_battery"  in sample_results.columns else 0,
    "EV":      sample_results["has_ev"].sum()       if "has_ev"       in sample_results.columns else 0,
    "AC":      sample_results["has_ac"].sum()       if "has_ac"       in sample_results.columns else 0,
    "HP":      sample_results["has_hp"].sum()       if "has_hp"       in sample_results.columns else 0,
}

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = ["#f4a261", "#2a9d8f", "#264653", "#e9c46a", "#e76f51"]
bars = ax.bar(rates.keys(), [v / n_total * 100 for v in rates.values()], color=colors, width=0.55)
for bar, n in zip(bars, rates.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{n}/{n_total}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Detection rate (%)")
ax.set_title(f"Appliance detection rates — {n_total} customers")
ax.set_ylim(0, max(rates.values()) / n_total * 100 * 1.25)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
fig.tight_layout()
plt.show()

In [ ]:
# ── PV capacity distribution for the sample ───────────────────────────────────
pv_only = sample_results.dropna(subset=["pv_capacity_kwp"])
pv_only = pv_only[pv_only["pv_capacity_kwp"] > 0]

if len(pv_only) > 2:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

    axes[0].hist(pv_only["pv_capacity_kwp"].clip(upper=30), bins=20,
                 color="#f4a261", edgecolor="white")
    axes[0].set_xlabel("Estimated PV capacity (kWp)")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"PV capacity — {len(pv_only)} customers")

    if "sc_share" in pv_only.columns:
        axes[1].hist(pv_only["sc_share"].dropna().clip(0, 1), bins=20,
                     color="#2a9d8f", edgecolor="white")
        axes[1].set_xlabel("Self-consumption share")
        axes[1].set_ylabel("Count")
        axes[1].set_title("Self-consumption distribution")

    fig.tight_layout()
    plt.show()
else:
    print("Too few PV customers in sample to plot — try increasing N_CUSTOMERS")

---
## 4  Full Portfolio

Two paths:
- **Path A** — Full pipeline run via the orchestrator (writes results to `data/processed/out/`).
- **Path B** — Load existing results (reference outputs from a prior run).

The plots below adapt to whichever results are available.

In [ ]:
# ── Path A: run the full pipeline ─────────────────────────────────────────────
# Set RUN_FULL_PIPELINE = True to run all customers (takes minutes to hours depending on N).
# Leave False to load existing reference results instead.

RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    import logging
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S")

    from re_nilm.pipeline.orchestrator import PipelineOrchestrator, load_config

    cfg = load_config(REPO / "config" / "re_production.yaml")
    cfg["output"]["results_dir"] = str(RESULTS_DIR)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    orch = PipelineOrchestrator(cfg, enabled_detectors=["pv", "battery", "ev"])
    orch._load_weather = lambda: weather   # skip re-fetching
    portfolio = orch.run()
    portfolio.to_parquet(RESULTS_FILE, index=False)
    print(f"Pipeline complete: {len(portfolio):,} customers")

In [ ]:
# ── Path B: load existing results ─────────────────────────────────────────────
if RESULTS_FILE.exists():
    portfolio = pd.read_parquet(RESULTS_FILE)
    print(f"Loaded pipeline results: {len(portfolio):,} customers")
elif REF_CAPACITY.exists():
    # Fall back to the reference capacity output from the old pipeline
    portfolio = pd.read_parquet(REF_CAPACITY)
    # Align column names to new schema where needed
    if "has_pv_prob" in portfolio.columns and "has_pv" not in portfolio.columns:
        portfolio["has_pv"] = portfolio["has_pv_prob"] == 1.0
    if "sc_share_mean" in portfolio.columns and "sc_share" not in portfolio.columns:
        portfolio["sc_share"] = portfolio["sc_share_mean"]
    print(f"Loaded reference results: {len(portfolio):,} customers")
else:
    print("No results found — run the pipeline (RUN_FULL_PIPELINE = True) or run Section 3 first.")
    portfolio = sample_results   # use the small sample as fallback

print("Columns:", portfolio.columns.tolist())

In [ ]:
# ── Portfolio summary ─────────────────────────────────────────────────────────
n = len(portfolio)
summary = {}
for col, label in [("has_pv", "PV"), ("has_battery", "Battery"), ("has_ev", "EV"),
                   ("has_ac", "AC"), ("has_hp", "HP")]:
    if col in portfolio.columns:
        k = portfolio[col].sum()
        summary[label] = {"n": int(k), "rate_%": round(k / n * 100, 1)}

print(f"Portfolio: {n:,} customers\n")
for label, s in summary.items():
    print(f"  {label:<10} {s['n']:>6,}  ({s['rate_%']}%)")

if "pv_capacity_kwp" in portfolio.columns:
    pv_valid = portfolio["pv_capacity_kwp"].dropna()
    pv_valid = pv_valid[pv_valid > 0]
    print(f"\nPV capacity (kWp): median={pv_valid.median():.1f}  "
          f"p10={pv_valid.quantile(0.1):.1f}  p90={pv_valid.quantile(0.9):.1f}")
    total_kwp = pv_valid.sum()
    print(f"Total installed PV: {total_kwp:,.0f} kWp  ({total_kwp/1000:.1f} MWp)")

In [ ]:
# ── Plot: appliance penetration rates ─────────────────────────────────────────
if summary:
    labels_all = list(summary.keys())
    rates_all  = [summary[l]["rate_%"] for l in labels_all]
    counts_all = [summary[l]["n"] for l in labels_all]

    fig, ax = plt.subplots(figsize=(7, 3.5))
    palette = ["#f4a261", "#2a9d8f", "#264653", "#e9c46a", "#e76f51"]
    bars = ax.bar(labels_all, rates_all, color=palette[:len(labels_all)], width=0.55)
    for bar, n_pos in zip(bars, counts_all):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{n_pos:,}", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("Penetration (%)")
    ax.set_title(f"Portfolio appliance penetration — {n:,} customers")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    fig.tight_layout()
    plt.show()

In [ ]:
# ── Plot: PV capacity distribution ────────────────────────────────────────────
if "pv_capacity_kwp" in portfolio.columns:
    pv_df = portfolio.dropna(subset=["pv_capacity_kwp"])
    pv_df = pv_df[pv_df["pv_capacity_kwp"] > 0]

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

    # Histogram
    axes[0].hist(pv_df["pv_capacity_kwp"].clip(upper=40), bins=40,
                 color="#f4a261", edgecolor="white", linewidth=0.4)
    axes[0].set_xlabel("Estimated PV capacity (kWp)")
    axes[0].set_ylabel("Number of customers")
    axes[0].set_title(f"PV capacity distribution  (n={len(pv_df):,})")
    med = pv_df["pv_capacity_kwp"].median()
    axes[0].axvline(med, color="#e63946", linestyle="--", label=f"Median {med:.1f} kWp")
    axes[0].legend()

    # Cumulative
    sorted_cap = pv_df["pv_capacity_kwp"].sort_values().values
    cumulative_mwp = sorted_cap.cumsum() / 1000
    axes[1].plot(sorted_cap, cumulative_mwp, color="#264653", linewidth=1.5)
    axes[1].set_xlabel("Individual capacity (kWp)")
    axes[1].set_ylabel("Cumulative installed (MWp)")
    axes[1].set_title("Cumulative PV capacity")

    fig.tight_layout()
    plt.show()

In [ ]:
# ── Plot: self-consumption share distribution ─────────────────────────────────
sc_col = "sc_share" if "sc_share" in portfolio.columns else "sc_share_mean"
if sc_col in portfolio.columns:
    sc = portfolio[sc_col].dropna()
    sc = sc[(sc >= 0) & (sc <= 1)]

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.hist(sc, bins=30, color="#2a9d8f", edgecolor="white", linewidth=0.4)
    ax.set_xlabel("Self-consumption share")
    ax.set_ylabel("Count")
    ax.set_title(f"Self-consumption distribution  (n={len(sc):,})")
    ax.axvline(sc.median(), color="#e63946", linestyle="--",
               label=f"Median {sc.median():.2f}")
    ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
# ── Plot: appliance co-occurrence (Venn-style bar) ────────────────────────────
# Which combinations of appliances co-occur?
combos_available = [("has_pv", "has_battery"), ("has_pv", "has_ev"), ("has_ev", "has_battery")]
combo_rows = []
for c1, c2 in combos_available:
    if c1 in portfolio.columns and c2 in portfolio.columns:
        both  = (portfolio[c1] & portfolio[c2]).sum()
        only1 = (portfolio[c1] & ~portfolio[c2]).sum()
        only2 = (~portfolio[c1] & portfolio[c2]).sum()
        combo_rows.append({"combo": f"{c1[4:].upper()} only", "n": int(only1)})
        combo_rows.append({"combo": f"{c2[4:].upper()} only", "n": int(only2)})
        combo_rows.append({"combo": f"Both",               "n": int(both)})

if combo_rows:
    combo_df = pd.DataFrame(combo_rows)
    print("\nAppliance co-occurrence:")
    display(combo_df.pivot_table(index="combo", values="n", aggfunc="sum"))

In [ ]:
# ── Portfolio-level capacity scatter (capacity vs yearly production) ───────────
if REF_INDICATORS.exists() and "pv_capacity_kwp" in portfolio.columns:
    indicators = pd.read_parquet(REF_INDICATORS)
    merged = portfolio.merge(
        indicators[["customer_id", "yearly_prod", "yearly_cons"]],
        on="customer_id", how="inner"
    ).dropna(subset=["pv_capacity_kwp", "yearly_prod"])
    merged = merged[(merged.pv_capacity_kwp > 0) & (merged.yearly_prod > 0)]

    fig, ax = plt.subplots(figsize=(7, 5))
    # Expected yield line: ~1000 kWh/kWp for Switzerland
    cap_range = np.linspace(0, merged.pv_capacity_kwp.quantile(0.99), 100)
    ax.plot(cap_range, cap_range * 1000, "--", color="#e63946", linewidth=1,
            label="Reference: 1000 kWh/kWp·year")

    ax.scatter(merged.pv_capacity_kwp, merged.yearly_prod,
               alpha=0.3, s=6, color="#264653", label="Customers")
    ax.set_xlabel("Estimated PV capacity (kWp)")
    ax.set_ylabel("Measured annual PV production (kWh)")
    ax.set_title(f"Capacity vs. production validation  (n={len(merged):,})")
    ax.set_xlim(0, merged.pv_capacity_kwp.quantile(0.99) * 1.05)
    ax.set_ylim(0)
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("Capacity vs. production scatter: need pv_indicators_clean.parquet")

---
## 5  PV Generation Forecast

For every PV-positive customer, project 15-min PV generation and net consumption.
Aggregate to get the portfolio-level load curve.

In [ ]:
from re_nilm.portfolio.forecasting import forecast_pv_for_customers, build_net_consumption

# Use a capped subsample to keep it fast in the notebook
MAX_FORECAST_CUSTOMERS = 500
forecast_df = portfolio.dropna(subset=["pv_capacity_kwp"])
forecast_df = forecast_df[forecast_df["pv_capacity_kwp"] > 0.1].head(MAX_FORECAST_CUSTOMERS)

print(f"Forecasting PV generation for {len(forecast_df)} customers...")
pv_fc = forecast_pv_for_customers(
    results=forecast_df,
    customer_index=index,
    weather_df=weather,
    efficiency=0.15,
)
print(f"Forecast rows: {len(pv_fc):,}")

In [ ]:
# ── Portfolio aggregate load curve ────────────────────────────────────────────
if not pv_fc.empty:
    net = build_net_consumption(pv_fc)

    # Daily averages for a summer week
    net["dt_utc"] = pd.to_datetime(net["dt_utc"])
    summer_week = net.set_index("dt_utc").loc["2024-07-08":"2024-07-14"]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.fill_between(summer_week.index,
                    summer_week["total_net_consumption_kwh"] * 4,  # kWh → kW
                    alpha=0.5, color="steelblue", label="Net consumption (kW)")
    ax.fill_between(summer_week.index,
                    summer_week["total_pv_kwh"] * 4,
                    alpha=0.6, color="gold", label="PV generation (kW)")
    ax.set_ylabel("Portfolio power (kW)")
    ax.set_title(f"Portfolio aggregate load — {summer_week['n_customers'].iloc[0]:,} PV customers  (July 2024)")
    ax.legend()
    fig.tight_layout()
    plt.show()

    # Average diurnal profile across the full year
    net["hour"] = net["dt_utc"].dt.hour + net["dt_utc"].dt.minute / 60
    diurnal = net.groupby("hour")[["total_pv_kwh", "total_net_consumption_kwh"]].mean() * 4

    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(diurnal.index, diurnal["total_pv_kwh"], color="gold", linewidth=2, label="Mean PV (kW)")
    ax.plot(diurnal.index, diurnal["total_net_consumption_kwh"], color="steelblue",
            linewidth=2, label="Mean net consumption (kW)")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Portfolio power (kW)")
    ax.set_title("Average diurnal profile across PV customers")
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("No forecast data — check that customer parquets cover the forecast period")

---
## CLI Reference

Everything above can also be run from the command line:

```bash
# Run full pipeline
python scripts/run_pipeline.py --config config/re_production.yaml

# Run specific detectors only
python scripts/run_pipeline.py --config config/re_production.yaml --detectors pv,battery,ev

# Start fresh (ignore checkpoints)
python scripts/run_pipeline.py --config config/re_production.yaml --no-resume

# Train AC + HP models
python scripts/train_models.py --training-data data/processed_data/all_sources_load_with_weather.parquet

# Export portfolio figures
python scripts/export_figures.py --config config/re_production.yaml
```